## 14. Conclusion & Resources

In this tutorial, we:
1. Loaded a HuggingFace model and applied LoRA adapters
2. Prepared a real instruction-tuning dataset (Dolly 15k)
3. Used torchax's functional training API (`JittableModule` + `make_train_step` + `optax`)
4. Trained on TPU with JIT-compiled training steps
5. Evaluated improvement with before/after comparison
6. Saved the fine-tuned model and reloaded it for inference

### Key takeaway

torchax lets you **keep your PyTorch model** while using **JAX's training primitives** — functional gradients, compiled training steps, and TPU acceleration. No JAX rewrite needed.

### Resources

- [torchax GitHub](https://github.com/google/torchax)
- [torchax PEFT example](https://github.com/google/torchax/blob/main/examples/peft_lora_training.py)
- [Inference tutorial (Part 1)](https://colab.research.google.com/github/ahmed-elnaggar/torchax-huggingface/blob/main/notebooks/torchax_huggingface_tutorial.ipynb)
- [Original tutorial series by Han Qi](https://huggingface.co/blog/qihqi/huggingface-jax-01)
- [PEFT/LoRA documentation](https://huggingface.co/docs/peft)
- [optax documentation](https://optax.readthedocs.io/)
- [JAX documentation](https://docs.jax.dev/)

### Credits

- **[Han Qi (@qihqi)](https://github.com/qihqi)** — author of torchax, PEFT training example, and the original tutorial series
- **[torchax team at Google](https://github.com/google/torchax)** — library development
- **[HuggingFace](https://huggingface.co/)** — transformers, PEFT, and datasets ecosystem
- **[Databricks](https://www.databricks.com/)** — Dolly 15k dataset
- **[JAX team at Google](https://github.com/jax-ml/jax)** — JAX, XLA, and TPU support

## 13. Troubleshooting

| Error | Cause | Fix |
|---|---|---|
| `OutOfMemoryError` | Model + gradients + optimizer too large | Switch to `TRAINING_MODE = "lora"`, reduce `BATCH_SIZE` to 1, or reduce `MAX_SEQ_LEN` |
| `TypeError: ... not a valid JAX type` | Custom HuggingFace type not registered as pytree | Register the type with `jax.tree_util.register_pytree_node()` |
| `Loss is NaN` | Learning rate too high or numerical instability | Reduce `LEARNING_RATE` by 10x, ensure `torch_dtype=torch.bfloat16` |
| `PEFT model incompatible with torchax` | LoRA layers use unsupported ops | Try different `LORA_TARGET_MODULES`, or switch to full fine-tuning |
| `make_train_step error` | API mismatch with installed torchax version | Update: `pip install -U torchax`, check `model_fn` signature matches `(weights, buffers, args)` |
| `Slow first training step` | Normal — JAX JIT compilation | Wait ~30-60s for the first step; subsequent steps are fast |
| `tokenizer.apply_chat_template error` | Model doesn't have a chat template | Use a manual template: `f"<start_of_turn>user\n{msg}<end_of_turn>\n<start_of_turn>model\n{response}<end_of_turn>"` |
| `Dataset download fails` | Network issues or dataset moved | Try `DATASET_NAME = "timdettmers/openassistant-guanaco"` as fallback |

### Memory Optimization Tips

1. **Switch to LoRA** — reduces trainable params by 99%+
2. **Use AdaFactor** — set `USE_ADAFACTOR = True` for ~50% less optimizer memory
3. **Reduce sequence length** — `MAX_SEQ_LEN = 256` halves activation memory
4. **Reduce batch size** — `BATCH_SIZE = 1` with higher `GRAD_ACCUM_STEPS`
5. **Limit training data** — `MAX_TRAIN_SAMPLES = 500` for quick experiments

## 12. Swapping Models & Datasets

To use a different model, simply change `MODEL_NAME` in the configuration cell:

```python
MODEL_NAME = "google/gemma-3-1b-it"     # Smaller, faster
MODEL_NAME = "google/gemma-2-2b-it"     # Gemma 2 variant
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Good for testing
```

To use a different dataset, change `DATASET_NAME`:

```python
DATASET_NAME = "tatsu-lab/alpaca"                    # Stanford Alpaca (52k examples)
DATASET_NAME = "timdettmers/openassistant-guanaco"   # Open Assistant (10k examples)
```

For larger models, consider:
- Reducing `MAX_SEQ_LEN` to 256
- Using LoRA mode (always recommended for free Colab)
- Reducing `MAX_TRAIN_SAMPLES` to 500-1000

In [ ]:
# Mini chat session with the fine-tuned model
def chat(user_message, max_new_tokens=100):
    """Chat with the fine-tuned model."""
    return generate_response(reloaded_model, tokenizer, user_message, device, max_new_tokens)

# Demo conversation
questions = [
    "Explain quantum computing to a 10-year-old.",
    "What are three creative ways to reduce food waste?",
    "Write a short haiku about artificial intelligence.",
]

for q in questions:
    print(f"User: {q}")
    response = chat(q)
    print(f"Model: {response}")
    print("-" * 60)

In [ ]:
# Verify: generate with the reloaded model
test_prompt = "What is the most important thing to consider when starting a new business?"
print(f"Q: {test_prompt}\n")
response = generate_response(reloaded_model, tokenizer, test_prompt, device, max_new_tokens=100)
print(f"A: {response}")

In [ ]:
# Free memory from training (if training was completed)
for var_name in ['params', 'buffers', 'opt_state', 'frozen_params']:
    if var_name in dir():
        del globals()[var_name]
import gc; gc.collect()

# Reload the model fresh with torchax disabled during loading
with tx.disable_temporarily():
    if TRAINING_MODE == "lora":
        # Load base model + LoRA adapters
        reloaded_model = transformers.AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, torch_dtype=torch.bfloat16
        )
        reloaded_model = peft.PeftModel.from_pretrained(reloaded_model, save_dir)
    else:
        reloaded_model = transformers.AutoModelForCausalLM.from_pretrained(
            save_dir, torch_dtype=torch.bfloat16
        )

reloaded_model.to(device)
reloaded_model.eval()
print("Reloaded model successfully!")

## 11. Reload & Run Inference

Let's verify the saved model works by loading it fresh and running inference — proving the full train → save → reload pipeline works.

In [ ]:
save_dir = "./fine_tuned_model"

with torch.no_grad():
    # Convert JAX arrays back to CPU tensors for saving
    cpu_state_dict = {
        name: torch.tensor(np.array(p)).contiguous()
        for name, p in params.items()
    }

    # Save using HuggingFace's standard format
    model.save_pretrained(save_dir, state_dict=cpu_state_dict)

tokenizer.save_pretrained(save_dir)

# Show what was saved
import os
saved_files = os.listdir(save_dir)
total_size = sum(os.path.getsize(os.path.join(save_dir, f)) for f in saved_files)
print(f"Model saved to: {save_dir}")
print(f"Files: {saved_files}")
print(f"Total size: {total_size / 1e6:.1f} MB")
if TRAINING_MODE == "lora":
    print("(Only LoRA adapter weights saved — base model loaded separately at inference time)")

## 10. Save the Fine-Tuned Model

We convert JAX arrays back to CPU tensors and save using HuggingFace's standard format. For LoRA, only the tiny adapter weights are saved (~20MB). For full fine-tuning, the entire model is saved (~4GB).

In [ ]:
# Qualitative comparison: before vs after fine-tuning
print("Generating fine-tuned responses for comparison...\n")
finetuned_responses = []
for prompt in eval_prompts:
    response = generate_response(model, tokenizer, prompt, device, max_new_tokens=80)
    finetuned_responses.append(response)

print("=" * 80)
print("SIDE-BY-SIDE COMPARISON: Before vs After Fine-Tuning")
print("=" * 80)
for i, prompt in enumerate(eval_prompts):
    print(f"\nQ: {prompt}")
    print(f"\nBefore: {baseline_responses[i][:200]}{'...' if len(baseline_responses[i]) > 200 else ''}")
    print(f"\nAfter:  {finetuned_responses[i][:200]}{'...' if len(finetuned_responses[i]) > 200 else ''}")
    print("-" * 80)

In [ ]:
# Load final trained params back into model for evaluation
with torch.no_grad():
    for name, param in params.items():
        try:
            parts = name.split(".")
            obj = model
            for part in parts[:-1]:
                obj = getattr(obj, part)
            setattr(obj, parts[-1], torch.nn.Parameter(param))
        except AttributeError as e:
            print(f"Warning: Could not load parameter '{name}': {e}")

# Final evaluation
print("Running final evaluation...")
final_loss, final_ppl = evaluate_loss(model, eval_dataloader, device, max_batches=50)

print(f"\n{'Metric':<20} {'Before':>10} {'After':>10} {'Change':>10}")
print(f"{'─' * 50}")
print(f"{'Loss':<20} {baseline_loss:>10.4f} {final_loss:>10.4f} {final_loss - baseline_loss:>+10.4f}")
print(f"{'Perplexity':<20} {baseline_ppl:>10.2f} {final_ppl:>10.2f} {(final_ppl - baseline_ppl) / baseline_ppl * 100:>+9.1f}%")

## 9. Post-Training Evaluation

Now let's measure the improvement from fine-tuning — both quantitatively (loss/perplexity) and qualitatively (generated responses).

### Reading the Training Curves

**Training loss (left):** Should decrease over time. The raw per-step loss (blue) is noisy; the smoothed curve (red) shows the trend. If loss plateaus early, try increasing the learning rate or training for more epochs.

**Eval perplexity (right):** Lower is better. The dashed line shows the baseline (before training). If the green line goes well below the baseline, fine-tuning is working. If it increases, the model may be overfitting — try reducing the number of training samples or epochs.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training loss curve
axes[0].plot(train_losses, alpha=0.3, color="#4285F4", label="Per-step loss")
# Smoothed curve (rolling average)
window = min(20, len(train_losses) // 5) if len(train_losses) > 5 else 1
if window > 1:
    smoothed = np.convolve(train_losses, np.ones(window) / window, mode="valid")
    axes[0].plot(range(window - 1, len(train_losses)), smoothed, color="#EA4335", linewidth=2, label=f"Smoothed (window={window})")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Eval perplexity over time
if eval_history:
    steps, losses, ppls = zip(*eval_history)
    axes[1].plot(steps, ppls, "o-", color="#34A853", linewidth=2, markersize=6)
    axes[1].axhline(y=baseline_ppl, color="#FBBC04", linestyle="--", label=f"Baseline ({baseline_ppl:.1f})")
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("Perplexity")
    axes[1].set_title("Eval Perplexity")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
else:
    axes[1].text(0.5, 0.5, "No eval checkpoints yet", ha="center", va="center", transform=axes[1].transAxes)

plt.tight_layout()
plt.show()

In [ ]:
import time
from tqdm.auto import tqdm

torch.manual_seed(SEED)
model.train()
step = 0
train_losses = []
eval_history = []

print(f"Starting training: {NUM_EPOCHS} epoch(s), {len(train_dataloader)} batches/epoch")
print(f"Gradient accumulation: {GRAD_ACCUM_STEPS} steps (effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS})")
print(f"Eval every {EVAL_EVERY_STEPS} steps\n")

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    pbar = tqdm(enumerate(train_dataloader), total=len(train_dataloader), desc=f"Epoch {epoch + 1}")

    for batch_idx, batch in pbar:
        batch = {k: v.to(device) for k, v in batch.items()}

        # Run one training step
        # step_fn signature: (weights, buffers, opt_state, args, labels) -> (loss, weights, opt_state)
        loss, params, opt_state = step_fn(
            params, buffers, opt_state, batch, batch["labels"]
        )

        loss_val = loss.item()
        train_losses.append(loss_val)
        step += 1
        pbar.set_postfix({"loss": f"{loss_val:.4f}", "step": step})

        # Periodic evaluation
        if step % EVAL_EVERY_STEPS == 0:
            # Temporarily load params back into model for evaluation
            with torch.no_grad():
                for name, param in params.items():
                    try:
                        parts = name.split(".")
                        obj = model
                        for part in parts[:-1]:
                            obj = getattr(obj, part)
                        setattr(obj, parts[-1], torch.nn.Parameter(param))
                    except AttributeError:
                        pass

            eval_loss, eval_ppl = evaluate_loss(model, eval_dataloader, device, max_batches=25)
            eval_history.append((step, eval_loss, eval_ppl))
            elapsed = time.time() - start_time
            print(f"\n  Step {step} | train_loss={loss_val:.4f} | eval_loss={eval_loss:.4f} | eval_ppl={eval_ppl:.2f} | time={elapsed:.0f}s")

elapsed = time.time() - start_time
print(f"\nTraining complete! {step} steps in {elapsed:.0f}s ({elapsed/60:.1f} min)")

## 8. Training Loop

Now we run the training loop. The first step will be slow (JIT compilation), but subsequent steps are fast.

**What to expect:**
- Step 1: ~30-60s (JAX compiles the training step)
- Steps 2+: ~1-3s each
- Total: ~20-40 minutes for 2000 samples with LoRA

In [ ]:
# Define the model function and loss function for make_train_step
# model_fn must be a pure function: (weights, buffers, args) -> output
def model_fn(weights, buffers, batch):
    """Stateless forward pass using functional_call."""
    return torch.func.functional_call(
        model, {**weights, **buffers}, args=(), kwargs=batch
    )

def loss_fn(model_output, labels):
    """Extract loss from HuggingFace model output."""
    return model_output.loss

# Compose into a single training step
# make_train_step handles: forward → loss → jax.value_and_grad → optax.update
step_fn = tx.train.make_train_step(model_fn, loss_fn, optimizer)

print("Training step function created.")
print("Pipeline: forward → loss → jax.value_and_grad → optax.update → apply_updates")

In [ ]:
# Create optax optimizer with learning rate schedule
lr = LEARNING_RATE if TRAINING_MODE == "lora" else LEARNING_RATE / 10
total_steps = NUM_EPOCHS * len(train_dataloader) // GRAD_ACCUM_STEPS

schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0,
    peak_value=lr,
    warmup_steps=WARMUP_STEPS,
    decay_steps=total_steps,
)

if TRAINING_MODE == "full" and USE_ADAFACTOR:
    optimizer = optax.adafactor(learning_rate=schedule)
    print("Optimizer: AdaFactor (memory-efficient for full fine-tuning)")
else:
    optimizer = optax.adamw(learning_rate=schedule, weight_decay=0.01)
    print(f"Optimizer: AdamW (lr={lr})")

# Initialize optimizer state using call_jax (bridges optax JAX call with torchax tensors)
opt_state = tx.interop.call_jax(optimizer.init, params)
print(f"Optimizer state initialized")
print(f"Total training steps: {total_steps}")

In [ ]:
import numpy as np
import torchax.train

# Separate trainable params from frozen buffers
# For LoRA: only adapter weights are trainable (requires_grad=True)
# For full fine-tuning: all parameters are trainable
params = {n: p for n, p in model.named_parameters() if p.requires_grad}
buffers = dict(model.named_buffers())
frozen_params = {n: p for n, p in model.named_parameters() if not p.requires_grad}
buffers.update(frozen_params)

if not params:
    raise RuntimeError(
        "No trainable parameters found! "
        "If using LoRA, check that LORA_TARGET_MODULES matches your model's layer names. "
        "If using full fine-tuning, ensure the model was loaded correctly."
    )

trainable_count = sum(p.numel() for p in params.values())
frozen_count = sum(p.numel() for p in frozen_params.values())
print(f"Trainable params: {trainable_count / 1e6:.2f}M")
print(f"Frozen params:    {frozen_count / 1e6:.2f}M")
print(f"Buffers:          {len(buffers)}")

## 7. Set Up Functional Training

This is where torchax training diverges from standard PyTorch. We:
1. **Separate** the model into trainable `params` and frozen `buffers`
2. **Create** an optax optimizer (functional, no hidden state)
3. **Define** a pure loss function for `jax.value_and_grad`
4. **Compose** everything into a single JIT-compiled training step

In [ ]:
# Generate baseline responses for qualitative comparison
def generate_response(model, tokenizer, instruction, device, max_new_tokens=100):
    """Generate a response to an instruction using greedy decoding."""
    try:
        messages = [{"role": "user", "content": instruction}]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt")
        input_ids = inputs["input_ids"].to(device)

        model.eval()
        generated_ids = input_ids.clone()
        with torch.no_grad():
            for _ in range(max_new_tokens):
                outputs = model(generated_ids, use_cache=False)
                next_token = torch.argmax(outputs.logits[:, -1, :], dim=-1, keepdim=True)
                generated_ids = torch.cat([generated_ids, next_token], dim=-1)
                if next_token.item() == tokenizer.eos_token_id:
                    break
        model.train()

        new_tokens = generated_ids[0, input_ids.shape[1]:]
        return tokenizer.decode(new_tokens, skip_special_tokens=True)
    except Exception as e:
        model.train()
        return f"[Generation failed: {e}]"

# Sample prompts for qualitative evaluation
eval_prompts = [
    "Explain what machine learning is in simple terms.",
    "What are the benefits of renewable energy?",
    "Write a short poem about the ocean.",
    "Summarize the key differences between Python and Java.",
    "Give me three tips for better time management.",
]

print("Generating baseline responses...")
baseline_responses = []
for prompt in eval_prompts:
    response = generate_response(model, tokenizer, prompt, device, max_new_tokens=80)
    baseline_responses.append(response)
    print(f"\nQ: {prompt}")
    print(f"A: {response[:200]}{'...' if len(response) > 200 else ''}")

In [ ]:
import math

def evaluate_loss(model, dataloader, device, max_batches=None):
    """Compute average loss and perplexity on a dataset."""
    model.eval()
    total_loss = 0.0
    total_batches = 0

    with torch.no_grad():
        for i, batch in enumerate(dataloader):
            if max_batches and i >= max_batches:
                break
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            total_loss += outputs.loss.item()
            total_batches += 1

    model.train()
    if total_batches == 0:
        print("Warning: No batches evaluated — dataloader may be empty.")
        return float("inf"), float("inf")
    avg_loss = total_loss / total_batches
    perplexity = math.exp(min(avg_loss, 100))  # Cap to avoid overflow
    return avg_loss, perplexity

# Run baseline evaluation
print("Running baseline evaluation...")
baseline_loss, baseline_ppl = evaluate_loss(model, eval_dataloader, device, max_batches=50)
print(f"Baseline loss:       {baseline_loss:.4f}")
print(f"Baseline perplexity: {baseline_ppl:.2f}")

## 6. Baseline Evaluation (Before Training)

Before training, we measure the model's performance on our eval set. This gives us a baseline to compare against after fine-tuning.

In [ ]:
# Enable torchax and move model to JAX device
tx.enable_globally()
device = torch.device("jax")
model = model.to(device)
model.train()

print(f"Model moved to: {device}")
print(f"JAX devices available: {jax.devices()}")

In [ ]:
# Conditionally apply LoRA
if TRAINING_MODE == "lora":
    peft_config = peft.LoraConfig(
        task_type=peft.TaskType.CAUSAL_LM,
        inference_mode=False,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODULES,
    )
    model = peft.get_peft_model(model, peft_config)
    model.print_trainable_parameters()
else:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Full fine-tuning: {trainable / 1e6:.1f}M / {total / 1e6:.1f}M params trainable")

In [ ]:
import torchax as tx

# Load model on CPU with torchax temporarily disabled
# This prevents torchax from intercepting unsupported initialization ops
with tx.disable_temporarily():
    model = transformers.AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.bfloat16
    )

print(f"Model loaded: {MODEL_NAME}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

## 5. Load Model & Apply LoRA

We load the model on CPU first (to avoid torchax intercepting unsupported init ops), optionally apply LoRA, then move to JAX.

In [ ]:
from torch.utils.data import DataLoader

# Use HuggingFace's data collator for causal language modeling
collator = transformers.DataCollatorForLanguageModeling(
    tokenizer=tokenizer, mlm=False
)

train_dataloader = DataLoader(
    train_tokenized, shuffle=True, collate_fn=collator, batch_size=BATCH_SIZE
)
eval_dataloader = DataLoader(
    eval_tokenized, shuffle=False, collate_fn=collator, batch_size=BATCH_SIZE
)

# Inspect a batch
sample_batch = next(iter(train_dataloader))
print("Batch keys:", list(sample_batch.keys()))
for k, v in sample_batch.items():
    print(f"  {k}: shape={v.shape}, dtype={v.dtype}")

In [ ]:
import os
import tempfile

# Tokenize the formatted examples
def tokenize_example(example):
    """Tokenize a formatted example with padding and truncation."""
    formatted = format_example(example)
    tokens = tokenizer(
        formatted["text"],
        padding="max_length",
        max_length=MAX_SEQ_LEN,
        truncation=True,
        return_tensors=None,
    )
    return tokens

train_tokenized = train_dataset.map(
    tokenize_example,
    load_from_cache_file=False,
    cache_file_name=os.path.join(tempfile.gettempdir(), "train_cache.arrow"),
    remove_columns=train_dataset.column_names,
)
eval_tokenized = eval_dataset.map(
    tokenize_example,
    load_from_cache_file=False,
    cache_file_name=os.path.join(tempfile.gettempdir(), "eval_cache.arrow"),
    remove_columns=eval_dataset.column_names,
)

print(f"Train tokenized: {len(train_tokenized)} examples")
print(f"Eval tokenized:  {len(eval_tokenized)} examples")
print(f"Token columns:   {train_tokenized.column_names}")
print(f"Sequence length: {len(train_tokenized[0]['input_ids'])}")

In [ ]:
from transformers import AutoTokenizer

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Subset and split into train/eval
subset = raw_dataset.shuffle(seed=SEED).select(range(min(MAX_TRAIN_SAMPLES + MAX_EVAL_SAMPLES, len(raw_dataset))))
split = subset.train_test_split(test_size=MAX_EVAL_SAMPLES, seed=SEED)
train_dataset = split["train"]
eval_dataset = split["test"]

print(f"Train samples: {len(train_dataset)}")
print(f"Eval samples:  {len(eval_dataset)}")

# Format into chat template
def format_example(example):
    """Format a dolly example into Gemma's chat template."""
    user_content = example["instruction"]
    if example.get("context", ""):
        user_content += f"\n\nContext: {example['context']}"

    messages = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": example["response"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

# Show a formatted example
sample = format_example(train_dataset[0])
print(f"\nFormatted example (first 300 chars):\n{sample['text'][:300]}...")

In [ ]:
import datasets as hf_datasets

# Load dataset (single "train" split — we split ourselves)
raw_dataset = hf_datasets.load_dataset(DATASET_NAME, split="train")
print(f"Dataset size: {len(raw_dataset)} examples")
print(f"Columns: {raw_dataset.column_names}")
print(f"\nExample:")
for key in ["instruction", "context", "response", "category"]:
    val = raw_dataset[0][key]
    print(f"  {key}: {val[:100]}{'...' if len(val) > 100 else ''}")

## 4. Load and Prepare the Dataset

We use [Databricks Dolly 15k](https://huggingface.co/datasets/databricks/databricks-dolly-15k) — 15,000 human-written instruction-response pairs across 7 categories (QA, summarization, brainstorming, etc.).

Each example has:
- `instruction` — the task to perform
- `context` — optional background information
- `response` — the expected output
- `category` — task type

We format these into Gemma's chat template for instruction tuning.

## 3. Key Concepts for Training

### JittableModule

`JittableModule` wraps a `torch.nn.Module` and separates it into two parts:
- **params** — trainable parameters (weights that get updated during training)
- **buffers** — everything else (frozen weights, running stats, constants)

This separation is required because JAX's `jax.value_and_grad` needs to know which inputs to differentiate. Only `params` get gradients.

### optax Optimizers

Unlike PyTorch optimizers (which carry hidden mutable state), JAX optimizers are **pure functions**:
```python
# PyTorch: hidden state inside optimizer
optimizer.step()

# optax: explicit state, no hidden pockets
updates, new_opt_state = optimizer.update(grads, opt_state, params)
new_params = optax.apply_updates(params, updates)
```

### donate_argnums

When updating weights, you normally need memory for both old and new params (2x peak). `donate_argnums` tells JAX: "I'm done with the old params — reuse that memory for the new ones." This halves peak memory.

*Analogy: Instead of copying a document, editing the copy, and throwing away the original — just edit the original directly.*

### Full Fine-Tuning vs LoRA

| | Full Fine-Tuning | LoRA |
|---|---|---|
| **Trainable params** | All (~2B) | Tiny adapters (~0.5%) |
| **Memory** | ~18-20 GB | ~5-7 GB |
| **Speed** | Slower | Faster |
| **Quality** | Higher ceiling | Nearly as good |
| **Free Colab TPU** | Tight / may OOM | Fits comfortably |

**LoRA** (Low-Rank Adaptation) freezes the base model and adds small trainable matrices to attention layers. Instead of updating the full weight matrix W, it learns a low-rank decomposition: `W + (α/r) × B·A` where A and B are tiny matrices.

In [ ]:
# Verify installation
import jax
import torch
import torchax
import transformers
import peft
import datasets
import optax

print(f"JAX version:          {jax.__version__}")
print(f"JAX devices:          {jax.devices()}")
print(f"PyTorch version:      {torch.__version__}")
print(f"torchax version:      {torchax.__version__}")
print(f"transformers version: {transformers.__version__}")
print(f"peft version:         {peft.__version__}")
print(f"datasets version:     {datasets.__version__}")
print(f"optax version:        {optax.__version__}")

In [ ]:
# Install PyTorch CPU (torchax handles the accelerator via JAX)
!pip install -q torch --index-url https://download.pytorch.org/whl/cpu

# Install JAX for the detected accelerator
if accelerator == "tpu":
    !pip install -q -U jax[tpu]
elif accelerator == "gpu":
    !pip install -q -U jax[cuda12]
else:
    !pip install -q -U jax

# Install torchax, transformers, and training dependencies
!pip install -q -U torchax transformers flax peft datasets optax

print("\nInstallation complete!")

In [ ]:
import subprocess
import sys

def detect_accelerator():
    """Detect available accelerator: TPU, GPU, or CPU."""
    try:
        import jax
        devices = jax.devices()
        if any(d.platform == "tpu" for d in devices):
            return "tpu"
        if any(d.platform == "gpu" for d in devices):
            return "gpu"
    except Exception:
        pass
    try:
        result = subprocess.run(["ls", "/dev/accel0"], capture_output=True)
        if result.returncode == 0:
            return "tpu"
    except Exception:
        pass
    try:
        result = subprocess.run(["nvidia-smi"], capture_output=True)
        if result.returncode == 0:
            return "gpu"
    except Exception:
        pass
    return "cpu"

accelerator = detect_accelerator()
print(f"Detected accelerator: {accelerator}")

## 2. Setup: Detect Runtime & Install Dependencies

In [ ]:
# ============================================================
# USER CONFIGURATION — Change these to customize your training
# ============================================================

MODEL_NAME = "google/gemma-4-E2B-it"
TRAINING_MODE = "lora"          # "full" or "lora"

# LoRA settings (ignored if TRAINING_MODE == "full")
LORA_RANK = 16
LORA_ALPHA = 32
LORA_TARGET_MODULES = ["q_proj", "v_proj"]
LORA_DROPOUT = 0.05

# Dataset
DATASET_NAME = "databricks/databricks-dolly-15k"
MAX_SEQ_LEN = 512
MAX_TRAIN_SAMPLES = 2000        # Subset for Colab time constraints
MAX_EVAL_SAMPLES = 200

# Training hyperparameters
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4            # Effective batch size = BATCH_SIZE * GRAD_ACCUM_STEPS
NUM_EPOCHS = 1
LEARNING_RATE = 2e-4            # Good default for LoRA; auto-adjusted for full fine-tuning
WARMUP_STEPS = 50
EVAL_EVERY_STEPS = 50
SEED = 42

# Full fine-tuning memory optimizations (ignored if TRAINING_MODE == "lora")
USE_GRADIENT_CHECKPOINTING = True
USE_ADAFACTOR = True            # AdaFactor uses ~50% less optimizer memory than AdamW

print(f"Training mode: {TRAINING_MODE}")
print(f"Model: {MODEL_NAME}")
print(f"Dataset: {DATASET_NAME}")
print(f"Max train samples: {MAX_TRAIN_SAMPLES}")

## 1. Background: Why Fine-Tune on TPUs?

In the [inference tutorial](https://colab.research.google.com/github/ahmed-elnaggar/torchax-huggingface/blob/main/notebooks/torchax_huggingface_tutorial.ipynb), we ran HuggingFace models on JAX for fast inference. But what about **training**?

Training on TPUs traditionally required rewriting your model in JAX (Flax, Equinox) or using PyTorch/XLA. **torchax** offers a third path: keep your PyTorch model, but use JAX's training primitives — `jax.grad` for gradients, `optax` for optimizers, and `jax.jit` for compiled training steps.

### How torchax Training Differs from Standard PyTorch

| Standard PyTorch | torchax |
|---|---|
| `loss.backward()` | `jax.value_and_grad(loss_fn)(params, ...)` |
| `optimizer.step()` | `optax.apply_updates(params, updates)` |
| Model holds its own state | Params and buffers are separate pytrees |
| Eager execution | JIT-compiled training steps |

*Analogy: If inference is like **reading a book**, training is like **studying with a highlighter**. torchax's `JittableModule` separates the pages (frozen buffers) from your highlighted notes (trainable params) so JAX knows exactly what to learn from.*

# Fine-Tune Any HuggingFace Model on TPUs with TorchAX

**A complete, beginner-friendly training tutorial**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmed-elnaggar/torchax-huggingface/blob/main/notebooks/torchax_training_tutorial.ipynb)

In this notebook, you will learn how to:
1. **Instruction-tune** a HuggingFace model on TPU using torchax
2. Choose between **full fine-tuning** and **LoRA** (parameter-efficient)
3. Use a **real dataset** (Databricks Dolly 15k) for training
4. **Evaluate** improvement with before/after comparison
5. **Save** the fine-tuned model and **reload** it for inference

We use [torchax](https://github.com/google/torchax), a library from Google that bridges PyTorch and JAX.

**Model:** `google/gemma-4-E2B-it` (2B params, fits on free Colab TPU with LoRA)

---

**Inference tutorial:** [Run Any HuggingFace Model on TPUs](https://colab.research.google.com/github/ahmed-elnaggar/torchax-huggingface/blob/main/notebooks/torchax_huggingface_tutorial.ipynb)

**Blog post:** [Fine-Tune Any HuggingFace Model on TPUs with TorchAX](https://dev.to/ahmed_elnaggar/fine-tune-any-huggingface-model-on-tpus-with-torchax)

**Credits:** Built on torchax's [PEFT LoRA training example](https://github.com/google/torchax/blob/main/examples/peft_lora_training.py) and the [3-part blog series](https://huggingface.co/blog/qihqi/huggingface-jax-01) by Han Qi ([@qihqi](https://github.com/qihqi)).